# Full-coverage soft canonical composition, one real pair

This notebook expands the current `match/config/partial_aware.yaml` defaults for one non-self FAUST pair. Despite the historical preset name, its default `matching_mode=full` requires every raw vertex of the second shape `N` to map to a real raw vertex of the first shape `M`. The original null-aware behavior remains available through `matching_mode=partial`.

```text
N raw ──barycentric──► canonical template ──target reverse map──► M raw
  │       A_n             optional K_T             Q_m             ▲
  └──────────── dense shared-topology fallback when evidence is empty ┘
                    P_(n→m) = A_n [K_T] Q_m
```

The notebook makes sparse barycentric canonicalization, confidence factors, full versus partial occupancy, dense fallback, soft composition, optional neutral-template smoothing, reciprocal consistency, geodesic evaluation, and production texture transfer visible stage by stage. It uses production helpers and checks the expanded default result against `PartialAwareSoftCanonizationMapExtractor.pipeline()`.

The canonical matcher itself remains independent of input-shape spectral data. This visualization additionally loads the Laplace–Beltrami operator, mass, and eigenpairs solely so `TextureTransferMetric(use_soft_permutation=True)` can spectrally soften each variant's final hard `p2p`. Gradients remain disabled. The optional geodesic matrix at the end is used only to evaluate the returned map.

## 1. Setup: LBO-free matching with spectral texture visualization

In [ ]:
import sys
from pathlib import Path

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "match").is_dir():
            return candidate
    fallback = Path("/home/knpob/work/code/atm-dev")
    if (fallback / "pyproject.toml").is_file():
        return fallback
    raise RuntimeError("Could not locate the atm-dev repository root.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
import scipy.sparse
import torch

from match.data.articulated_shape_corr import ArticulatedShapeCorrDataset
from match.map_extractor.partial_aware_soft_map_extractor import (
    PartialAwareSoftCanonizationMapExtractor,
    compose_soft_map,
    dense_canonical_fallback_map,
    hard_map_from_soft,
    load_smooth_kernel_cache,
    reverse_canonical_map,
)
from match.metric.geodist import GeodistMetric, _geodist_batch
from match.metric.texture_transfer import TextureTransferMetric

# These values mirror match/config/partial_aware.yaml.
DATA_ROOT = Path("/home/knpob/work/data/shape-corr/FAUST_r")
ARTICULATED_SHAPE_DIR = PROJECT_ROOT / "output/faust_r_uncertainty"
PAIR_INDEX = 1  # test Cartesian product: tr_reg_081 (N) → tr_reg_080 (M)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DISTANCE_SCALE = 0.02
UNCERTAINTY_SCALE = 0.02
KAPPA_FACTOR = 1.0
EPS = 1e-8
MATCHING_MODE = "full"
USE_VISIBILITY_TERM = False
USE_SMOOTH_KERNEL = False
USE_RECIPROCAL_CONSISTENCY = False
SMOOTH_KERNEL_CACHE_BASE_PATH = PROJECT_ROOT / "output/cache/MHR_smooth_kernel.npy"
SMOOTH_KERNEL_MHR_PATH = PROJECT_ROOT / "pkg/sam_3d_body/checkpoints/sam-3d-body-dinov3/assets/mhr_model.pt"
SMOOTH_KERNEL_SIGMA_FACTOR = 0.01
SMOOTH_KERNEL_MIN_WEIGHT = 1e-3
SMOOTH_KERNEL_BATCH_SIZE = 128
NUM_EVECS = 200
TEXTURE_PATH = PROJECT_ROOT / "docs/assets/texture-symbol-grid.png"
TEXTURE_TRANSFER_OUT_DIR = ARTICULATED_SHAPE_DIR / "notebook/texture-transfer"
PYVISTA_BACKEND = "static"  # Use "html" for rotatable scenes.

required = [
    DATA_ROOT / "off",
    DATA_ROOT / "corres",
    DATA_ROOT / "spectral",
    ARTICULATED_SHAPE_DIR / "mhr",
    TEXTURE_PATH,
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, (
    "Missing required assets. Run uncertainty-aware articulation for the pair first: "
    + ", ".join(missing)
)

pv.set_jupyter_backend(PYVISTA_BACKEND)
pv.global_theme.background = "white"
pv.global_theme.font.color = "black"

print(f"Dataset:       {DATA_ROOT}")
print(f"Articulation:  {ARTICULATED_SHAPE_DIR / 'mhr'}")
print(f"Device:        {DEVICE}")
print(f"Matching mode: {MATCHING_MODE}")
print(f"Texture:       {TEXTURE_PATH}")

In [ ]:
def to_numpy(value):
    if torch.is_tensor(value):
        return value.detach().cpu().numpy()
    return np.asarray(value)

def as_polydata(vertices, faces):
    vertices = to_numpy(vertices)
    faces = to_numpy(faces).astype(np.int64, copy=False)
    pv_faces = np.column_stack([np.full(len(faces), 3), faces]).ravel()
    return pv.PolyData(vertices, pv_faces)

def add_scalar_mesh(plotter, shape, values, title, *, cmap="viridis", clim=None):
    mesh = as_polydata(shape["verts"], shape["faces"])
    mesh.point_data[title] = np.asarray(values)
    plotter.add_mesh(
        mesh,
        scalars=title,
        cmap=cmap,
        clim=clim,
        smooth_shading=True,
        show_scalar_bar=True,
    )
    plotter.add_title(title, font_size=10)
    plotter.camera_position = "xy"

def add_proxy_scalar_mesh(plotter, shape, values, title, *, cmap="viridis", clim=None):
    mesh = as_polydata(shape["articulated_verts"], shape["articulated_faces"])
    mesh.point_data[title] = np.asarray(values)
    plotter.add_mesh(
        mesh,
        scalars=title,
        cmap=cmap,
        clim=clim,
        smooth_shading=True,
        show_scalar_bar=True,
    )
    plotter.add_title(title, font_size=10)
    plotter.camera_position = "xy"

def torch_sparse_to_scipy(matrix):
    matrix = matrix.coalesce().cpu()
    indices = matrix.indices().numpy()
    values = matrix.values().numpy()
    return scipy.sparse.csr_matrix(
        (values, (indices[0], indices[1])),
        shape=tuple(matrix.shape),
    )

def spatial_rgb(vertices):
    xyz = to_numpy(vertices).astype(np.float64)
    lo, hi = xyz.min(axis=0), xyz.max(axis=0)
    rgb = (xyz - lo) / np.maximum(hi - lo, 1e-12)
    return rgb[:, [0, 2, 1]]

def add_rgb_mesh(plotter, shape, rgb, title):
    mesh = as_polydata(shape["verts"], shape["faces"])
    mesh.point_data["RGB"] = np.clip(np.round(255 * rgb), 0, 255).astype(np.uint8)
    plotter.add_mesh(mesh, scalars="RGB", rgb=True, smooth_shading=True)
    plotter.add_title(title, font_size=10)
    plotter.camera_position = "xy"

## 2. Load one pair with texture-transfer spectral objects

The partial-aware matcher does not require spectral objects, but the production metric's `use_soft_permutation=True` path requires eigenvectors and mass-weighted transposes. This notebook therefore loads the LBO, mass, and 200 eigenpairs solely for texture visualization; gradients remain disabled. `return_dist=True` loads a geodesic distance matrix only for the final evaluation cell. Neither spectral nor distance data participates in canonical map construction.

In [ ]:
data_component = ArticulatedShapeCorrDataset(
    dataset_name="faust_r",
    data_root=str(DATA_ROOT),
    articulated_shape_dir=str(ARTICULATED_SHAPE_DIR),
    articulated_subdir="mhr",
    phase="test",
    return_faces=True,
    return_L=True,
    return_mass=True,
    num_evecs=NUM_EVECS,
    return_evecs=True,
    return_grad=False,
    return_corr=True,
    return_dist=True,  # evaluation only
    return_articulation_confidence=True,
)
pair_dataset = data_component.pipeline()
first_index, second_index = pair_dataset.combinations[PAIR_INDEX]
pair = pair_dataset[PAIR_INDEX]
shape_m = pair["first"]   # real target columns
shape_n = pair["second"]  # query/source rows

spectral_keys = {"L", "mass", "evecs", "evecs_trans", "evals", "gradX", "gradY"}
required_spectral_keys = {"L", "mass", "evecs", "evecs_trans", "evals"}
assert required_spectral_keys.issubset(shape_m)
assert required_spectral_keys.issubset(shape_n)
assert {"gradX", "gradY"}.isdisjoint(shape_m)
assert {"gradX", "gradY"}.isdisjoint(shape_n)
assert shape_m["evecs"].shape[1] == NUM_EVECS
assert shape_n["evecs"].shape[1] == NUM_EVECS
assert shape_m["articulated_faces"].shape == shape_n["articulated_faces"].shape
assert torch.equal(shape_m["articulated_faces"], shape_n["articulated_faces"])

print(f"Direction: {shape_n['name']} (N) → {shape_m['name']} (M)")
print(f"Raw vertices:       N={len(shape_n['verts']):,}, M={len(shape_m['verts']):,}")
print(f"Canonical vertices: {len(shape_n['articulated_verts']):,}")
print("Loaded spectral keys:", sorted(spectral_keys.intersection(shape_m)))

In [ ]:
plotter = pv.Plotter(shape=(1, 2), window_size=(1400, 650))
for column, (shape, label) in enumerate([(shape_n, "N / query"), (shape_m, "M / target")]):
    plotter.subplot(0, column)
    plotter.add_mesh(
        as_polydata(shape["verts"], shape["faces"]),
        color="white",
        opacity=0.5,
        label="raw",
        smooth_shading=True,
    )
    plotter.add_mesh(
        as_polydata(shape["articulated_verts"], shape["articulated_faces"]),
        color="salmon",
        opacity=0.6,
        label="articulated proxy",
        smooth_shading=True,
    )
    plotter.add_title(f"{label}: raw + shared-topology proxy", font_size=10)
    plotter.camera_position = "xy"
plotter.link_views()
plotter.add_legend()
plotter.show()

## 3. Sparse barycentric canonicalization $A_i$

For raw vertex $x$, the closest point on its articulated proxy lies in one proxy triangle. Its three barycentric coordinates become one row of $A_i\in\mathbb{R}^{N_i\times M}$. Rows sum to one, while explicit zeros on triangle edges or vertices are removed. This preserves sub-triangle location instead of snapping every raw point to one canonical vertex.

The closest-triangle query is the expensive part. We call the production component's per-shape canonicalization method once; its cache is then reused by the end-to-end equivalence check.

In [ ]:
extractor = PartialAwareSoftCanonizationMapExtractor(
    device=DEVICE,
    distance_scale=DISTANCE_SCALE,
    uncertainty_scale=UNCERTAINTY_SCALE,
    kappa_factor=KAPPA_FACTOR,
    eps=EPS,
    cache_canonicalization=True,
    matching_mode=MATCHING_MODE,
    use_visibility_term=USE_VISIBILITY_TERM,
    use_smooth_kernel=USE_SMOOTH_KERNEL,
    use_reciprocal_consistency=USE_RECIPROCAL_CONSISTENCY,
    smooth_kernel_cache_path=SMOOTH_KERNEL_CACHE_BASE_PATH,
    smooth_kernel_mhr_path=SMOOTH_KERNEL_MHR_PATH,
    smooth_kernel_sigma_factor=SMOOTH_KERNEL_SIGMA_FACTOR,
    smooth_kernel_min_weight=SMOOTH_KERNEL_MIN_WEIGHT,
    smooth_kernel_batch_size=SMOOTH_KERNEL_BATCH_SIZE,
)

canonical_m = extractor._canonicalize(shape_m)
canonical_n = extractor._canonicalize(shape_n)
A_m = canonical_m["assignment"]
A_n = canonical_n["assignment"]

for label, assignment, shape in [("M", A_m, shape_m), ("N", A_n, shape_n)]:
    row_sums = np.asarray(assignment.sum(axis=1)).ravel()
    np.testing.assert_allclose(row_sums, 1.0, atol=1e-10)
    assert assignment.shape == (
        len(shape["verts"]), len(shape["articulated_verts"])
    )
    print(
        f"A_{label.lower()}: shape={assignment.shape}, nnz={assignment.nnz:,}, "
        f"nnz/row={assignment.nnz / assignment.shape[0]:.3f}"
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for ax, assignment, label in [(axes[0], A_n, "N"), (axes[1], A_m, "M")]:
    entries_per_row = np.diff(assignment.indptr)
    counts = np.bincount(entries_per_row, minlength=4)
    ax.bar(np.arange(len(counts)), counts, color="#4c78a8")
    ax.set_xticks(np.arange(len(counts)))
    ax.set_xlabel("stored barycentric entries per raw vertex")
    ax.set_ylabel("raw vertex count")
    ax.set_title(f"Sparsity of A_{label.lower()}")
plt.show()

## 4. Canonicalization confidence and the disabled visibility term

For each raw point $x$,

$$
q_i(x)=
q_i^{\mathrm{distance}}(x)\,
q_i^{\mathrm{uncertainty}}(x)\,
q_i^{\mathrm{normal}}(x)\,
q_i^{\mathrm{visibility}}(x).
$$

The first factor penalizes raw-to-proxy distance, the second interpolates articulation variance through $A_i$, and the third checks normal agreement. The raw weighted view support is still interpolated and retained as `visibility_support`, but the complete-shape default sets the effective $q_i^{\mathrm{visibility}}=1$ so three retained articulation views cannot erase otherwise valid target evidence. Set `use_visibility_term=true` to restore the original partial-aware product. These are reliability scores, not calibrated probabilities.

In [ ]:
diagnostics_m = canonical_m["diagnostics"]
diagnostics_n = canonical_n["diagnostics"]
factor_names = [
    "distance_term", "uncertainty_term", "normal_term", "visibility_term"
]

for label, diagnostics in [("M", diagnostics_m), ("N", diagnostics_n)]:
    if not USE_VISIBILITY_TERM:
        np.testing.assert_allclose(diagnostics["visibility_term"], 1.0)
    product = np.ones_like(diagnostics["confidence"])
    for name in factor_names:
        product *= diagnostics[name]
    np.testing.assert_allclose(product, diagnostics["confidence"], atol=1e-12)
    print(
        f"{label} confidence: mean={diagnostics['confidence'].mean():.4f}, "
        f"zero={(diagnostics['confidence'] == 0).mean():.3f}, "
        f"max={diagnostics['confidence'].max():.4f}, "
        f"raw visibility support mean={diagnostics['visibility_support'].mean():.4f}"
    )

fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
for row, (diagnostics, label) in enumerate([(diagnostics_n, "N / query"), (diagnostics_m, "M / target")]):
    for name in factor_names:
        axes[row, 0].hist(
            diagnostics[name], bins=50, histtype="step", linewidth=1.5, label=name
        )
    axes[row, 0].hist(
        diagnostics["visibility_support"],
        bins=50,
        histtype="step",
        linewidth=1.5,
        linestyle="--",
        label="visibility_support (diagnostic)",
    )
    axes[row, 0].set_xlim(0, 1)
    axes[row, 0].set_title(f"{label}: confidence factors")
    axes[row, 0].set_xlabel("factor value")
    axes[row, 0].set_ylabel("raw vertex count")
    axes[row, 0].legend(fontsize=8)

    axes[row, 1].hist(diagnostics["confidence"], bins=60, color="#e15759")
    axes[row, 1].set_xlim(0, 1)
    axes[row, 1].set_title(f"{label}: product q")
    axes[row, 1].set_xlabel("canonicalization confidence")
plt.show()

In [ ]:
plotter = pv.Plotter(shape=(1, 4), window_size=(2000, 520))
confidence_panels = [
    (shape_n, diagnostics_n["distance"], "N: proxy distance", "magma"),
    (shape_n, diagnostics_n["confidence"], "N: combined confidence", "viridis"),
    (shape_m, diagnostics_m["distance"], "M: proxy distance", "magma"),
    (shape_m, diagnostics_m["confidence"], "M: combined confidence", "viridis"),
]
for column, (shape, values, title, cmap) in enumerate(confidence_panels):
    plotter.subplot(0, column)
    clim = [0.0, float(np.quantile(values, 0.99))] if "distance" in title else [0.0, 1.0]
    add_scalar_mesh(plotter, shape, values, title, cmap=cmap, clim=clim)
plotter.link_views()
plotter.show()

## 5. Target evidence and full occupancy in $Q_m$

Target raw vertices contribute evidence in proportion to their incident surface area $\omega_{mj}$, confidence $q_m(y_j)$, and barycentric assignment:

$$
d_m(u)=\sum_j \omega_{mj}q_m(y_j)A_m(j,u).
$$

In partial mode, $o_m(u)=1-\exp(-d_m(u)/\kappa)$ and the last column of $Q_m\in\mathbb{R}^{M\times(N_m+1)}$ stores $1-o_m(u)$. The complete-shape default instead forces $o_m(u)=1$, so null mass is zero everywhere. Evidence-free reverse rows can still be empty; the later dense fallback handles those rows explicitly rather than misrepresenting sparse articulation confidence as missing geometry.

In [ ]:
Q_m, target_evidence, target_occupancy, kappa = reverse_canonical_map(
    A_m,
    canonical_m["area_weights"],
    diagnostics_m["confidence"],
    total_surface_area=canonical_m["total_area"],
    kappa_factor=KAPPA_FACTOR,
    force_full_occupancy=MATCHING_MODE == "full",
    eps=EPS,
)

expected_kappa = KAPPA_FACTOR * canonical_m["total_area"] / A_m.shape[1]
np.testing.assert_allclose(kappa, expected_kappa)
assert Q_m.shape == (A_m.shape[1], len(shape_m["verts"]) + 1)

q_row_sums = np.asarray(Q_m.sum(axis=1)).ravel()
assert np.all(q_row_sums <= 1.0 + 1e-10)
if MATCHING_MODE == "full":
    np.testing.assert_allclose(target_occupancy, 1.0)
    np.testing.assert_allclose(Q_m[:, -1].toarray(), 0.0)

print(f"Target surface area: {canonical_m['total_area']:.6f}")
print(f"kappa:               {kappa:.6e}")
print(
    "Canonical occupancy: "
    f"min={target_occupancy.min():.4f}, "
    f"median={np.median(target_occupancy):.4f}, "
    f"max={target_occupancy.max():.4f}"
)

plotter = pv.Plotter(shape=(1, 3), window_size=(1650, 560))
canonical_panels = [
    (target_evidence, "target evidence d_m", "magma", [0.0, float(np.quantile(target_evidence, 0.99))]),
    (target_occupancy, "target occupancy o_m", "viridis", [0.0, 1.0]),
    (1.0 - target_occupancy, "canonical null mass", "inferno", [0.0, 1.0]),
]
for column, (values, title, cmap, clim) in enumerate(canonical_panels):
    plotter.subplot(0, column)
    add_proxy_scalar_mesh(plotter, shape_m, values, title, cmap=cmap, clim=clim)
plotter.link_views()
plotter.show()

## 6. Soft composition $P_{n\to m}=A_nQ_m$

Composition retains confidence-weighted real mass through the shared canonical topology. $P_{n\to m}$ has one row per raw vertex of $N$, one column per real vertex of $M$, and one structurally retained null column. In full mode, the hard map takes the best available real candidate without comparing it to null. A row with no real evidence follows a guaranteed dense fallback: nearest source proxy vertex $\to$ the same shared-topology target proxy vertex $\to$ nearest target raw vertex. Partial mode instead keeps the original best-real-versus-null rejection rule.

In [ ]:
P_n2m = (A_n @ Q_m).tocsr()
P_n2m.eliminate_zeros()
P_real = P_n2m[:, : len(shape_m["verts"])].tocsr()
p2p_manual, best_real_manual, null_manual = hard_map_from_soft(
    P_n2m,
    len(shape_m["verts"]),
    allow_null_matches=MATCHING_MODE == "partial",
)
fallback_manual = dense_canonical_fallback_map(
    to_numpy(shape_n["verts"]),
    to_numpy(shape_n["articulated_verts"]),
    to_numpy(shape_m["articulated_verts"]),
    to_numpy(shape_m["verts"]),
)
fallback_used_manual = np.zeros(len(shape_n["verts"]), dtype=bool)
if MATCHING_MODE == "full":
    fallback_used_manual = p2p_manual < 0
    p2p_manual[fallback_used_manual] = fallback_manual[fallback_used_manual]
    assert np.all(p2p_manual >= 0)

# The end-to-end production call reuses the canonicalization cache populated above.
production = extractor.pipeline(shape_m=shape_m, shape_n=shape_n)
P_production = torch_sparse_to_scipy(production["soft_map"])

# Compare sparse values without allocating a dense 5k × 5k matrix.
sparse_difference = (P_n2m - P_production).tocoo()
max_sparse_error = (
    float(np.max(np.abs(sparse_difference.data)))
    if sparse_difference.nnz
    else 0.0
)
assert max_sparse_error < 1e-7
assert production["matching_mode"] == MATCHING_MODE
np.testing.assert_array_equal(p2p_manual, to_numpy(production["p2p"]))
np.testing.assert_array_equal(
    fallback_used_manual, to_numpy(production["dense_fallback_used"])
)
np.testing.assert_allclose(
    target_evidence, to_numpy(production["target_evidence"]), atol=1e-7
)
np.testing.assert_allclose(
    target_occupancy, to_numpy(production["target_occupancy"]), atol=1e-7
)
np.testing.assert_allclose(
    best_real_manual, to_numpy(production["best_real_probability"]), atol=1e-7
)
np.testing.assert_allclose(
    null_manual, to_numpy(production["null_probability"]), atol=1e-7
)

accepted = p2p_manual >= 0
coverage = float(accepted.mean())
print(f"P shape:       {P_n2m.shape}")
print(f"P nonzeros:    {P_n2m.nnz:,}")
print(f"Hard coverage: {coverage:.3f} ({accepted.sum():,}/{len(accepted):,})")
print(
    f"Dense fallback: {fallback_used_manual.mean():.3f} "
    f"({fallback_used_manual.sum():,}/{len(fallback_used_manual):,})"
)
print("Expanded sparse composition exactly matches production pipeline().")

In [ ]:
# A binned sparse preview avoids densifying the full soft map.
preview_size = 512
preview = np.zeros((preview_size, preview_size), dtype=np.float32)
coo = P_n2m.tocoo()
preview_rows = np.minimum(
    preview_size - 1, coo.row * preview_size // P_n2m.shape[0]
)
preview_cols = np.minimum(
    preview_size - 1, coo.col * preview_size // P_n2m.shape[1]
)
np.maximum.at(preview, (preview_rows, preview_cols), coo.data)

supported_rows = np.flatnonzero(best_real_manual > 0)
fallback_rows = np.flatnonzero(fallback_used_manual)
row_examples = {
    "strongest real evidence": int(np.argmax(best_real_manual)),
    "weakest nonzero evidence": int(
        supported_rows[np.argmin(best_real_manual[supported_rows])]
    ),
    "dense fallback": int(
        fallback_rows[0] if len(fallback_rows) else np.argmin(best_real_manual)
    ),
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4.2), constrained_layout=True)
axes[0].imshow(np.log10(preview + 1e-8), origin="lower", cmap="magma", aspect="auto")
axes[0].set_xlabel("M real vertices + null (binned)")
axes[0].set_ylabel("N query vertices (binned)")
axes[0].set_title("log10 sparse P occupancy")

for ax, (label, row_id) in zip(axes[1:], row_examples.items()):
    row = P_real.getrow(row_id)
    order = np.argsort(row.data)[-8:][::-1]
    candidate_ids = row.indices[order]
    candidate_values = row.data[order]
    bars = [f"M:{index}" for index in candidate_ids] + ["NULL"]
    values = np.r_[candidate_values, null_manual[row_id]]
    colors = ["#4c78a8"] * len(candidate_values) + ["#e15759"]
    ax.bar(np.arange(len(values)), values, color=colors)
    ax.set_xticks(np.arange(len(values)), bars, rotation=70, fontsize=7)
    ax.set_title(
        f"{label}\nN vertex {row_id}, hard={p2p_manual[row_id]}"
    )
    ax.set_ylabel("soft mass")
plt.show()

## 7. Optional neutral-MHR geodesic smoothing $K_T$

With `use_smooth_kernel=true`, production inserts a row-normalized sparse Gaussian kernel between $A_n$ and $Q_m$. The first call builds it from a zero-identity, zero-expression, neutral-pose MHR template; later calls load the compressed CSR cache. Dijkstra searches stop at the radius implied by the pruning threshold, so the dense $18{,}439^2$ geodesic matrix is never materialized.

In [ ]:
smooth_extractor = PartialAwareSoftCanonizationMapExtractor(
    device=DEVICE,
    distance_scale=DISTANCE_SCALE,
    uncertainty_scale=UNCERTAINTY_SCALE,
    kappa_factor=KAPPA_FACTOR,
    eps=EPS,
    cache_canonicalization=True,
    matching_mode=MATCHING_MODE,
    use_visibility_term=USE_VISIBILITY_TERM,
    use_smooth_kernel=True,
    use_reciprocal_consistency=False,
    smooth_kernel_cache_path=SMOOTH_KERNEL_CACHE_BASE_PATH,
    smooth_kernel_mhr_path=SMOOTH_KERNEL_MHR_PATH,
    smooth_kernel_sigma_factor=SMOOTH_KERNEL_SIGMA_FACTOR,
    smooth_kernel_min_weight=SMOOTH_KERNEL_MIN_WEIGHT,
    smooth_kernel_batch_size=SMOOTH_KERNEL_BATCH_SIZE,
)
K_T = smooth_extractor._get_smooth_kernel(A_n.shape[1])
smooth_kernel_cache_path = smooth_extractor.smooth_kernel_cache_path
cached_K_T, kernel_cache_metadata = load_smooth_kernel_cache(
    smooth_kernel_cache_path
)
kernel_difference = (K_T - cached_K_T).tocoo()
assert kernel_difference.nnz == 0 or np.max(np.abs(kernel_difference.data)) < 1e-8
np.testing.assert_allclose(
    np.asarray(K_T.sum(axis=1)).ravel(), 1.0, rtol=1e-5, atol=1e-6
)
assert K_T.shape == (A_n.shape[1], A_n.shape[1])
assert K_T.nnz < K_T.shape[0] * K_T.shape[1]

kernel_anchor = int(np.argmax(target_occupancy))
kernel_row = K_T.getrow(kernel_anchor).toarray().ravel()
nonzero_kernel_row = kernel_row[kernel_row > 0]
plotter = pv.Plotter(window_size=(780, 700))
add_proxy_scalar_mesh(
    plotter,
    shape_n,
    kernel_row,
    f"K_T row at canonical vertex {kernel_anchor}",
    cmap="magma",
    clim=[0.0, float(np.quantile(nonzero_kernel_row, 0.99))],
)
plotter.show()

print(f"K_T shape:          {K_T.shape}")
print(f"K_T nonzeros:       {K_T.nnz:,}")
print(f"nonzeros per row:   {K_T.nnz / K_T.shape[0]:.1f}")
print(f"cache path:         {smooth_kernel_cache_path}")
print(f"compressed cache:   {smooth_kernel_cache_path.stat().st_size / 2**20:.1f} MiB")
print(f"cache metadata:     {kernel_cache_metadata}")

## 8. Compare optional smoothing and reciprocity under the full-map contract

The four variants use the same pair, full occupancy, disabled visibility term, and canonicalizations. Smoothing changes the forward soft map through $A_nK_TQ_m$. Reciprocal consistency computes the backward map and symmetric score $S_{nm}=\sqrt{P_{n\to m}(n,m)P_{m\to n}(m,n)}$ while leaving the forward soft map untouched. In full mode, non-mutual or evidence-free rows are replaced by the dense shared-topology fallback, so every final hard map remains total; the fallback mask makes those replacements visible.

In [ ]:
smooth_manual = compose_soft_map(A_n, Q_m, K_T)
smooth_only = smooth_extractor.pipeline(shape_m=shape_m, shape_n=shape_n)
smooth_production = torch_sparse_to_scipy(smooth_only["soft_map"])
smooth_difference = (smooth_manual - smooth_production).tocoo()
assert smooth_difference.nnz == 0 or np.max(np.abs(smooth_difference.data)) < 1e-7

reciprocal_extractor = PartialAwareSoftCanonizationMapExtractor(
    device=DEVICE,
    distance_scale=DISTANCE_SCALE,
    uncertainty_scale=UNCERTAINTY_SCALE,
    kappa_factor=KAPPA_FACTOR,
    eps=EPS,
    cache_canonicalization=True,
    matching_mode=MATCHING_MODE,
    use_visibility_term=USE_VISIBILITY_TERM,
    use_smooth_kernel=False,
    use_reciprocal_consistency=True,
    smooth_kernel_cache_path=SMOOTH_KERNEL_CACHE_BASE_PATH,
    smooth_kernel_mhr_path=SMOOTH_KERNEL_MHR_PATH,
    smooth_kernel_sigma_factor=SMOOTH_KERNEL_SIGMA_FACTOR,
    smooth_kernel_min_weight=SMOOTH_KERNEL_MIN_WEIGHT,
    smooth_kernel_batch_size=SMOOTH_KERNEL_BATCH_SIZE,
)
reciprocal_only = reciprocal_extractor.pipeline(
    shape_m=shape_m, shape_n=shape_n
)
reciprocal_forward = torch_sparse_to_scipy(reciprocal_only["soft_map"])
reciprocal_soft_difference = (P_n2m - reciprocal_forward).tocoo()
assert reciprocal_soft_difference.nnz == 0 or np.max(np.abs(reciprocal_soft_difference.data)) < 1e-7
np.testing.assert_array_equal(
    to_numpy(reciprocal_only["p2p_pre_reciprocal"]), p2p_manual
)

# Reuse the already loaded K_T and canonicalization cache for the combined run.
smooth_extractor.use_reciprocal_consistency = True
combined = smooth_extractor.pipeline(shape_m=shape_m, shape_n=shape_n)
smooth_extractor.use_reciprocal_consistency = False
combined_forward = torch_sparse_to_scipy(combined["soft_map"])
combined_difference = (smooth_manual - combined_forward).tocoo()
assert combined_difference.nnz == 0 or np.max(np.abs(combined_difference.data)) < 1e-7

variant_results = {
    "baseline": production,
    "smooth": smooth_only,
    "reciprocal": reciprocal_only,
    "smooth + reciprocal": combined,
}

def variant_hard_statistics(result):
    point_map = to_numpy(result["p2p"]).astype(np.int64)
    accepted_mask = point_map >= 0
    counts = np.bincount(point_map[accepted_mask], minlength=len(shape_m["verts"]))
    fallback_mask = to_numpy(result["dense_fallback_used"]).astype(bool)
    reciprocal_consistent = result.get("reciprocal_consistent")
    return {
        "coverage": float(accepted_mask.mean()),
        "accepted": int(accepted_mask.sum()),
        "many_to_one_excess": int(np.maximum(counts - 1, 0).sum()),
        "dense_fallbacks": int(fallback_mask.sum()),
        "reciprocal_inconsistent": (
            int((~to_numpy(reciprocal_consistent).astype(bool)).sum())
            if reciprocal_consistent is not None
            else 0
        ),
    }

variant_statistics = {
    name: variant_hard_statistics(result)
    for name, result in variant_results.items()
}
variant_statistics

In [ ]:
plotter = pv.Plotter(shape=(1, 4), window_size=(2000, 520))
for column, (name, result) in enumerate(variant_results.items()):
    plotter.subplot(0, column)
    fallback_variant = to_numpy(result["dense_fallback_used"]).astype(float)
    add_scalar_mesh(
        plotter,
        shape_n,
        fallback_variant,
        f"{name}: dense fallback used",
        cmap="coolwarm",
        clim=[0.0, 1.0],
    )
plotter.link_views()
plotter.show()

plotter = pv.Plotter(shape=(1, 2), window_size=(1200, 560))
plotter.subplot(0, 0)
add_scalar_mesh(
    plotter,
    shape_n,
    to_numpy(reciprocal_only["reciprocal_consistent"]).astype(float),
    "reciprocal-only mutual selection",
    cmap="coolwarm",
    clim=[0.0, 1.0],
)
plotter.subplot(0, 1)
add_scalar_mesh(
    plotter,
    shape_n,
    to_numpy(reciprocal_only["reciprocal_score"]),
    "selected reciprocal score",
    cmap="magma",
)
plotter.link_views()
plotter.show()

names = list(variant_statistics)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].bar(names, [variant_statistics[name]["dense_fallbacks"] for name in names])
axes[0].set_ylabel("dense fallback vertices")
axes[0].tick_params(axis="x", rotation=25)
axes[1].bar(names, [variant_statistics[name]["many_to_one_excess"] for name in names])
axes[1].set_ylabel("many-to-one excess selections")
axes[1].tick_params(axis="x", rotation=25)
plt.show()

assert all(stats["coverage"] == 1.0 for stats in variant_statistics.values())

## 9. Production texture transfer with `texture-symbol-grid.png`

`TextureTransferMetric` treats `shape_m` as the texture source and renders `shape_n` beside it. For this full-coverage comparison, each call deliberately starts from the variant's final hard `p2p` map—including dense fallback and reciprocal filtering—instead of `soft_map_real`, then enables `use_soft_permutation=True` to project that map through the 200 loaded LBO eigenvectors. The four calls use `docs/assets/texture-symbol-grid.png` for the default, smoothing-only, reciprocal-only, and smoothing-plus-reciprocal results. Each screenshot is written to its own ignored run-output directory and displayed in a 2×2 comparison.

In [ ]:
rgb_m = spatial_rgb(shape_m["verts"])
rgb_n_soft = np.asarray(P_real @ rgb_m)
rgb_n_hard = rgb_m[to_numpy(production["p2p"]).astype(np.int64)]

plotter = pv.Plotter(shape=(1, 4), window_size=(2000, 520))
plotter.subplot(0, 0)
add_rgb_mesh(plotter, shape_m, rgb_m, "M target colors")
plotter.subplot(0, 1)
add_rgb_mesh(plotter, shape_n, rgb_n_soft, "N production soft-map transfer")
plotter.subplot(0, 2)
add_rgb_mesh(plotter, shape_n, rgb_n_hard, "N final hard-p2p transfer")
plotter.subplot(0, 3)
add_scalar_mesh(
    plotter,
    shape_n,
    fallback_used_manual.astype(float),
    "N dense fallback used",
    cmap="coolwarm",
    clim=[0.0, 1.0],
)
plotter.link_views()
plotter.show()

texture_screenshots = {}
for variant_name, result in variant_results.items():
    assert bool(torch.all(result["p2p"] >= 0))
    hard_texture_result = dict(result)
    hard_texture_result["soft_map_real"] = None
    variant_slug = variant_name.replace(" + ", "-").replace(" ", "-")
    variant_out_dir = TEXTURE_TRANSFER_OUT_DIR / variant_slug
    texture_metric = TextureTransferMetric(
        out_dir=str(variant_out_dir),
        texture_path=str(TEXTURE_PATH),
        use_soft_permutation=True,
        camera_position="xy",
        translation=[1, 0, 0],
        window_size=[1600, 800],
        zoom=1.0,
        keep_mesh=False,
    )
    assert texture_metric.pipeline(hard_texture_result) == "exported"
    screenshot = (
        variant_out_dir
        / result["method"]
        / f"{shape_m['name']}--{shape_n['name']}.png"
    )
    assert screenshot.is_file()
    texture_screenshots[variant_name] = screenshot

fig, axes = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True)
for ax, (variant_name, screenshot) in zip(axes.ravel(), texture_screenshots.items()):
    ax.imshow(plt.imread(screenshot))
    fallback_count = variant_statistics[variant_name]["dense_fallbacks"]
    ax.set_title(f"{variant_name} — dense fallbacks: {fallback_count:,}")
    ax.axis("off")
plt.show()

for variant_name, screenshot in texture_screenshots.items():
    print(f"{variant_name:>19}: {screenshot}")

## 10. Evaluate the dense map and verify full coverage

The same geodesic metric supports both contracts: partial mode masks `-1` and reports accepted-only error with coverage, while full mode must accept every ground-truth sample. The default result below is asserted to have 100% coverage; fallback locations and the spatial error field reveal where completeness required the lower-confidence topology path.

In [ ]:
p2p = production["p2p"]
shape_m_device = production["shape_m"]
shape_n_device = production["shape_n"]

distances, gt_accepted = _geodist_batch(
    dist_x=shape_m_device["dist"][None],
    corr_x=shape_m_device["corr"][None],
    corr_y=shape_n_device["corr"][None],
    p2p_y2x=p2p[None],
)
matched = int(gt_accepted.sum())
total = int(gt_accepted.numel())
manual_geodist = float(distances[gt_accepted].mean()) if matched else float("nan")
manual_gt_coverage = matched / total if total else 0.0

metric = GeodistMetric(out_dir=str(ARTICULATED_SHAPE_DIR), export_path=None)
metric_output = metric.pipeline(
    {
        **production,
        "dataset_name": data_component.dataset_name,
    }
)
assert np.isclose(metric.values[0], manual_geodist)
assert np.isclose(metric.coverages[0], manual_gt_coverage)
if MATCHING_MODE == "full":
    assert manual_gt_coverage == 1.0

error_on_n = np.full(len(shape_n["verts"]), np.nan, dtype=np.float64)
corr_n = to_numpy(shape_n["corr"]).astype(np.int64)
gt_accepted_1d = to_numpy(gt_accepted[0]).astype(bool)
error_on_n[corr_n[gt_accepted_1d]] = to_numpy(distances[0, gt_accepted[0]])

plotter = pv.Plotter(shape=(1, 2), window_size=(1200, 560))
plotter.subplot(0, 0)
add_scalar_mesh(
    plotter,
    shape_n,
    fallback_used_manual.astype(float),
    "dense fallback used",
    cmap="coolwarm",
    clim=[0.0, 1.0],
)
plotter.subplot(0, 1)
finite = error_on_n[np.isfinite(error_on_n)]
add_scalar_mesh(
    plotter,
    shape_n,
    error_on_n,
    "geodesic error (full map)",
    cmap="magma",
    clim=[0.0, float(np.quantile(finite, 0.98))],
)
plotter.link_views()
plotter.show()

print(metric_output)

def evaluate_variant(result):
    variant_distances, variant_accepted = _geodist_batch(
        dist_x=result["shape_m"]["dist"][None],
        corr_x=result["shape_m"]["corr"][None],
        corr_y=result["shape_n"]["corr"][None],
        p2p_y2x=result["p2p"][None],
    )
    matched = int(variant_accepted.sum())
    total = int(variant_accepted.numel())
    error = (
        float(variant_distances[variant_accepted].mean())
        if matched
        else float("nan")
    )
    return {"geodesic_error": error, "gt_coverage": matched / total if total else 0.0}

variant_evaluation = {
    name: evaluate_variant(result) for name, result in variant_results.items()
}
evaluation_names = list(variant_evaluation)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].bar(
    evaluation_names,
    [variant_evaluation[name]["geodesic_error"] for name in evaluation_names],
)
axes[0].set_ylabel("geodesic error")
axes[0].tick_params(axis="x", rotation=25)
axes[1].bar(
    evaluation_names,
    [variant_evaluation[name]["gt_coverage"] for name in evaluation_names],
)
axes[1].set_ylim(0.0, 1.0)
axes[1].set_ylabel("ground-truth coverage")
axes[1].tick_params(axis="x", rotation=25)
plt.show()
variant_evaluation

In [ ]:
diagnostics = {
    "pair_n_to_m": f"{shape_n['name']} -> {shape_m['name']}",
    "method": production["method"],
    "matching_mode": production["matching_mode"],
    "visibility_term_enabled": production["visibility_term_enabled"],
    "input_spectral_keys": sorted(spectral_keys.intersection(shape_m)),
    "A_n_shape": A_n.shape,
    "A_n_nnz": A_n.nnz,
    "A_m_shape": A_m.shape,
    "A_m_nnz": A_m.nnz,
    "soft_map_shape": P_n2m.shape,
    "soft_map_nnz": P_n2m.nnz,
    "hard_map_coverage_all_vertices": coverage,
    "dense_fallback_count": int(fallback_used_manual.sum()),
    "dense_fallback_fraction": float(fallback_used_manual.mean()),
    "geodesic_error_gt": manual_geodist,
    "geodesic_gt_coverage": manual_gt_coverage,
    "null_probability_range": [float(null_manual.min()), float(null_manual.max())],
    "smooth_kernel_shape": K_T.shape,
    "smooth_kernel_nnz": K_T.nnz,
    "smooth_kernel_cache_path": str(smooth_kernel_cache_path),
    "smooth_kernel_cache_mib": smooth_kernel_cache_path.stat().st_size / 2**20,
    "variant_hard_statistics": variant_statistics,
    "variant_evaluation": variant_evaluation,
    "texture_path": str(TEXTURE_PATH),
    "texture_use_soft_permutation": True,
    "texture_num_evecs": NUM_EVECS,
    "texture_screenshots": {
        name: str(path) for name, path in texture_screenshots.items()
    },
    "reciprocal_soft_map_preserved": True,
    "production_equivalence": True,
}
diagnostics

## Visual correctness checklist

- Raw and articulated meshes should overlap in pose and location for both shapes; shared articulated faces must be exactly equal.
- Every row of $A_n$ and $A_m$ must sum to one and contain at most three stored barycentric entries.
- Distance, articulation uncertainty, and normal disagreement must only reduce confidence. With visibility disabled, `visibility_term` must be one while raw `visibility_support` remains inspectable.
- In full mode, target occupancy must be one and canonical null mass zero everywhere; target evidence may still expose unsupported reverse rows.
- The matrix direction must remain $N\to M$: rows index `shape_n`, real columns index `shape_m`, and the final column is null.
- Full mode must return no `-1`: rows without real soft evidence must be marked by `dense_fallback_used` and follow source raw $\to$ source proxy $\to$ target proxy $\to$ target raw direction.
- Every row of $K_T$ must sum to one, its neighborhood must stay local on the shared canonical topology, and its stored entry count must remain below the dense matrix size.
- Manual $A_nK_TQ_m$ must match smooth production composition, and reciprocal-only mode must preserve the baseline forward soft map exactly.
- Reciprocal inconsistency may increase dense fallback use but must not reduce final full-mode coverage; inspect both the mutual-selection field and fallback mask.
- All four production texture screenshots must use the symbol-grid texture on `shape_m`, start from each variant's final full-coverage `p2p` (including reciprocal filtering and dense fallback), and apply the metric's 200-eigenvector soft permutation on `shape_n`.
- Geodesic coverage must be exactly one in full mode, and the spatial error field should be inspected together with fallback locations.
- Manual $A_nQ_m$, hard decisions, dense fallback, evidence, occupancy, and probabilities must match the production extractor within the asserted tolerances.